In [ ]:
from Utils import TempRel_Utils
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding
from Reader import obtain_combined_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
datasets, label_list, label2id, id2label = obtain_combined_dataset(["TempEval3", "MAVEN", "TBDense"], "TempRel")

In [ ]:
# datasets["train"].to_json(".\\cleandata\\combined\\TempRel\\train.json")
# datasets["test"].to_json(".\\cleandata\\combined\\TempRel\\test.json")
# datasets["eval"].to_json(".\\cleandata\\combined\\TempRel\\eval.json")

In [ ]:
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

seps = ["<e1>", "</e1>", "<e2>", "</e2>", "<e>", "</e>", "<timex", "</timex>", "TIMEVAL=", "TYPE=DATE>", "TYPE=TIME>", "TYPE=DURATION>", "TYPE=SET>", "TYPE=UNKOWN>"]
tokenizer.add_tokens(seps)
model.resize_token_embeddings(len(tokenizer))
utils = TempRel_Utils(tokenizer, label2id, id2label)

In [ ]:
datasets = utils.tokenize_datasets(datasets)

In [ ]:
datasets

In [ ]:
import numpy as np
values, counts = np.unique(datasets["train"]['label'], return_counts=True)

In [ ]:
values

In [ ]:
counts

In [ ]:
training_args = TrainingArguments(
    output_dir="./results/TempRel",
    logging_dir="./logs",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    save_steps=5000,
    eval_steps=20000,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    metric_for_best_model="f1"
)

In [ ]:
event_timex_temprel = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt"),
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

In [ ]:
hist = event_timex_temprel.train()

In [ ]:
event_timex_temprel.save_model("./results/TempRel")
event_timex_temprel.tokenizer.save_pretrained("./results/TempRel")

In [ ]:
event_timex_temprel.evaluate(datasets["test"])